In [ ]:
!cp "/content/drive/MyDrive/Hate Speech LLMs/ARR Data/Human Validation/humanData.xlsx" .

In [ ]:
import numpy as np
import pandas as pd
import ast
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score

In [ ]:
df = pd.read_excel("humanData.xlsx")

In [ ]:
ref_col = 'Hop_2_Target_Basis_of_Identity (Gold)'
pred_col = 'Hop_2_Target_Basis_of_Identity (Gemini)'

In [ ]:
df = df[df["Hop_1_Triage (Gold)"] != "بی‌ضرر"]
df[pred_col] = df[pred_col].fillna('[]')

In [ ]:
refIdentities = []
for refIdentity in df[ref_col]:
  refIdentities.extend(ast.literal_eval(refIdentity))
set(refIdentities)

{'other',
 'جنسیت',
 'حرفه/شغل',
 'دین/باور',
 'سوابق تحصیلی/دانشگاهی',
 'قومیت/ملیت',
 'هویت فردی',
 'وابستگی سازمانی/نهادی',
 'وضعیت مهاجرت',
 'ویژگی ذهنی/رفتاری',
 'گرایش سیاسی'}

In [ ]:
llmIdentities = []
for llmIdentity in df[pred_col]:
  if llmIdentity == '':
    llmIdentities.extend([])
  else:
    llmIdentities.extend(ast.literal_eval(llmIdentity))
set(llmIdentities)

{'other',
 'جنسیت',
 'حرفه/شغل',
 'دین/باور',
 'سوابق تحصیلی/دانشگاهی',
 'قومیت/ملیت',
 'نقش اجتماعی/خانوادگی',
 'هویت فردی',
 'وابستگی سازمانی/نهادی',
 'وضعیت مهاجرت',
 'ویژگی ذهنی/رفتاری',
 'گرایش سیاسی'}

In [ ]:
len(set(refIdentities))

11

In [ ]:
len(set(llmIdentities))

12

In [ ]:
df[ref_col] = df[ref_col].apply(ast.literal_eval)
df[pred_col] = df[pred_col].apply(ast.literal_eval)

In [ ]:
df[ref_col] = df[ref_col].apply(
    lambda x: [label for label in x if label != 'other']
)

df[pred_col] = df[pred_col].apply(
    lambda x: [label for label in x if label != 'other']
)

In [ ]:
mlb = MultiLabelBinarizer()

mlb.fit(list(df[ref_col]) + list(df[pred_col]))

y_true = mlb.transform(df[ref_col])
y_pred = mlb.transform(df[pred_col])

In [ ]:
f1_micro = f1_score(y_true, y_pred, average='micro')
f1_macro = f1_score(y_true, y_pred, average='macro')

print(f"Micro F1: {f1_micro:.5f}")
print(f"Macro F1: {f1_macro:.5f}")

Micro F1: 0.88152
Macro F1: 0.79141
